# Benchmark Instance: RCdp208 (pop_size=30, max_iter=1000, runs=30)


In [ ]:
import os, shutil
REPO_DIR = '/tmp/ph-showoa'
if os.path.exists(REPO_DIR):
    !cd {REPO_DIR} && git fetch origin && git reset --hard origin/main
else:
    !git clone --depth 1 https://github.com/Welkie/ph-showoa.git {REPO_DIR}
if os.path.exists('/kaggle/working/ph-showoa'):
    shutil.rmtree('/kaggle/working/ph-showoa', ignore_errors=True)


In [ ]:
!nvidia-smi
import torch
print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("Kaggle GPU runtime is required: enable a CUDA accelerator before running the benchmark.")
torch.cuda.set_device(0)
print("Device name:", torch.cuda.get_device_name(0))
print("CUDA capability:", torch.cuda.get_device_capability(0))


In [ ]:
import torch
if not torch.cuda.is_available():
    raise RuntimeError("CUDA GPU is required for the Python/PyTorch CUDA solver.")
torch.cuda.set_device(0)
print("Python/PyTorch CUDA backend ready")
print("PyTorch version:", torch.__version__)
print("Device:", torch.cuda.get_device_name(0))
print("Capability:", torch.cuda.get_device_capability(0))
print("Implementation: Python + PyTorch CUDA tensors")


In [ ]:
import glob
import os
import re
import subprocess
import sys
import time as T
from collections import deque
import pandas as pd

DATASETS = ["rcdp208"]
CWD = "/tmp/ph-showoa" if os.path.exists("/tmp/ph-showoa") else ("ph-showoa" if os.path.exists("ph-showoa") else ".")
DATASET_DIR = "/kaggle/input/datasets/keith1101/ph-showoa/Wang_Chen"
OUTPUT_CSV = "/kaggle/working/summary_python_pytorch_cuda.csv"
LOG_DIR = "/kaggle/working/logs_python_pytorch_cuda"
RUNS = 30
MAX_ITER = 1000
POP_SIZE = 30
INIT_MODE = "sa_rcrs_grasp"
COMPUTE_BACKEND = "cuda"
ARCHITECTURE = "python_cuda"
WORKERS = 1
RESUME = True
MAX_TIMEOUT_S = None
TAIL_LINES = 20
os.makedirs(LOG_DIR, exist_ok=True)

def find_file(name):
    nc = name.strip().lower()
    candidates = [
        os.path.join(CWD, "dataset", f"explicit_{nc}.vrpsdptw"),
        os.path.join(DATASET_DIR, f"explicit_{nc}.vrpsdptw"),
    ]
    for pattern in (f"/kaggle/input/**/explicit_{nc}.vrpsdptw", f"**/explicit_{nc}.vrpsdptw"):
        candidates.extend(glob.glob(pattern, recursive=True))
    found = next((path for path in candidates if os.path.isfile(path)), None)
    if found:
        return found
    for root, dirs, files in os.walk("/kaggle/input"):
        for f in files:
            if f.lower() == f"explicit_{nc}.vrpsdptw" or f.lower() == f"{nc}.vrpsdptw":
                return os.path.join(root, f)
    for root, dirs, files in os.walk(CWD):
        for f in files:
            if f.lower() == f"explicit_{nc}.vrpsdptw" or f.lower() == f"{nc}.vrpsdptw":
                return os.path.join(root, f)
    return None

def parse_output(text):
    if "Traceback (most recent call last)" in text or "RuntimeError:" in text:
        return None
    runs = re.search(r"Total (\d+) runs, total consumed (\d+) sec", text)
    nv = re.search(r"(?:Vehicle count|vehicle \(route\) number):\s*(\d+)", text)
    cost = re.search(r"Total cost:\s*([\d.]+)", text)
    distance = re.search(r"Total distance:\s*([\d.]+)", text)
    if not (runs and nv and cost) or int(runs.group(1)) != RUNS:
        return None
    total_runs = int(runs.group(1))
    total_cost = float(cost.group(1))
    total_distance = float(distance.group(1)) if distance else total_cost - 2000.0 * int(nv.group(1))
    return {
        "best_NV": int(nv.group(1)),
        "best_TD": f"{total_distance:.4f}",
        "total_cost": f"{total_cost:.4f}",
        "avg_time_s": f"{float(runs.group(2)) / total_runs:.2f}",
        "total_runs": total_runs,
        "Status": "Success",
    }

def save_summary(rows):
    columns = ["Dataset", "best_NV", "best_TD", "total_cost", "avg_time_s", "wall_time", "total_runs", "Status", "Error"]
    frame = pd.DataFrame(rows)
    for column in columns:
        if column not in frame:
            frame[column] = "N/A"
    frame = frame[columns]
    frame.columns = ["Dataset", "Best NV", "Best TD", "Total Cost", "Avg/Run", "Wall Time", "Runs", "Status", "Error"]
    frame.to_csv(OUTPUT_CSV, index=False)

results = []
for name in DATASETS:
    problem = find_file(name)
    if problem is None:
        print(f"[ERROR] File for dataset {name} not found!")
        results.append({"Dataset": name, "Status": "File Not Found", "Error": "dataset file not found"})
        save_summary(results)
        continue

    command = [
        sys.executable, "-u", "-m", "src_python_gpu_SA_RCRS_GRASP.main",
        "--problem", problem, "--compute_backend", COMPUTE_BACKEND,
        "--init", INIT_MODE, "--paper_flags", "--architecture", ARCHITECTURE,
        "--objective", "lexicographic", "--grasp_alpha_lo", "0.10",
        "--grasp_alpha_hi", "0.40", "--sa_iterations", "25",
        "--runs", str(RUNS), "--max_iter", str(MAX_ITER),
        "--pop_size", str(POP_SIZE), "--workers", str(WORKERS),
    ]
    print(f"\n[RUN] {name}: Python/PyTorch CUDA | target runs={RUNS} pop_size={POP_SIZE} max_iter={MAX_ITER}")
    run_started = T.time()
    last_report = run_started
    lines = []
    tail = deque(maxlen=TAIL_LINES)
    timed_out = False
    process = subprocess.Popen(command, cwd=CWD, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    try:
        for line in process.stdout:
            lines.append(line)
            clean = line.rstrip()
            tail.append(clean)
            now = T.time()
            run_marker = re.search(r"Run (\d+).*", clean)
            total_marker = re.search(r"Total (\d+) runs, total consumed", clean)
            if run_marker and "Run " in clean:
                print(f"  [{now - run_started:.1f}s] {clean}", flush=True)
            elif total_marker:
                print(f"  [{now - run_started:.1f}s] {clean}", flush=True)
            elif now - last_report >= 20:
                print(f"  [{now - run_started:.0f}s] running... last: {clean[:100]}", flush=True)
                last_report = now
            if MAX_TIMEOUT_S and now - run_started > MAX_TIMEOUT_S:
                timed_out = True
                process.kill()
                break
        process.wait()
    except Exception:
        process.kill()
        process.wait()
        raise

    wall = T.time() - run_started
    output = "".join(lines)
    with open(os.path.join(LOG_DIR, f"{name}.log"), "w", encoding="utf-8") as log:
        log.write(output)
    return_code = process.returncode
    parsed = parse_output(output) if return_code == 0 and not timed_out else None
    if parsed is None:
        error_lines = [line.strip() for line in lines if "error" in line.lower() or "traceback" in line.lower()]
        row = {"Dataset": name, "Status": "Timeout" if timed_out else "Failed", "Error": " | ".join(error_lines[-3:]) or f"Python process exit code {return_code}", "wall_time": f"{wall:.1f}s"}
        print(f"[FAILED] {name}: {row['Error']}")
    else:
        row = parsed | {"Dataset": name, "wall_time": f"{wall:.1f}s", "Error": ""}
        print(f"[OK] {name}: completed {parsed['total_runs']}/{RUNS} runs | NV={row['best_NV']} TD={row['best_TD']} Avg/Run={row['avg_time_s']}s")
    results.append(row)
    save_summary(results)

print(f"Finished. Results saved to {OUTPUT_CSV}")


In [ ]:
import os
import pandas as pd
OUTPUT_CSV = "/kaggle/working/summary_python_pytorch_cuda.csv"
if os.path.exists(OUTPUT_CSV):
    df = pd.read_csv(OUTPUT_CSV)
    print("SUMMARY RESULT:")
    print(df.to_string(index=False))
